# TDR Analysis

Comprehensive evaluation of MCQ and Recon models on the TDR benchmark.

In [1]:
import json
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.metrics import f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

## Load Data

In [2]:
# Load benchmark
benchmark = [json.loads(l) for l in open('../data/benchmark.jsonl')]
print(f'Benchmark entries: {len(benchmark)}')

# Load homograph metadata
metadata = json.load(open('../data/homograph_metadata.json'))
print(f'Homographs: {len(metadata)}')

# Build lookup maps
ASYMMETRY_MAP = {h: info['asymmetry_type'] for h, info in metadata.items()}
BIGKAS_TAGS_MAP = {h: info['bigkas_tags'] for h, info in metadata.items()}

ASYMMETRY_TIERS = {
    'Workable': [h for h, a in ASYMMETRY_MAP.items() if a == 'Workable'],
    'Moderate': [h for h, a in ASYMMETRY_MAP.items() if a == 'Moderate'],
    'Extreme': [h for h, a in ASYMMETRY_MAP.items() if a == 'Extreme'],
}

BIGKAS_GROUPS = defaultdict(list)
for h, tags in BIGKAS_TAGS_MAP.items():
    for tag in tags:
        BIGKAS_GROUPS[tag].append(h)

print(f'\nAsymmetry tiers: {({k: len(v) for k, v in ASYMMETRY_TIERS.items()})}')
print(f'Bigkas groups: {({k: len(v) for k, v in BIGKAS_GROUPS.items()})}')

Benchmark entries: 7718
Homographs: 40

Asymmetry tiers: {'Workable': 13, 'Moderate': 7, 'Extreme': 20}
Bigkas groups: {'mabilis': 27, 'malumay': 35, 'maragsa': 7, 'malumi': 2}


In [3]:
# Load MCQ answer files
MCQ_MODELS = [
    'sealion-finetuned', 'sealion-vanilla',
    'llama-finetuned', 'llama-vanilla',
    'roberta-base', 'roberta-large',
    'roberta-tagalog-base', 'roberta-tagalog-large',
]

mcq_answers = {}
for model in MCQ_MODELS:
    path = f'../outputs/mcq/{model}.jsonl'
    try:
        mcq_answers[model] = [json.loads(l) for l in open(path)]
        print(f'MCQ {model}: {len(mcq_answers[model])} entries')
    except FileNotFoundError:
        print(f'MCQ {model}: NOT FOUND')

MCQ sealion-finetuned: 7718 entries
MCQ sealion-vanilla: 7718 entries


MCQ llama-finetuned: 7718 entries


MCQ llama-vanilla: 7718 entries
MCQ roberta-base: 7718 entries
MCQ roberta-large: 7718 entries


MCQ roberta-tagalog-base: 7718 entries
MCQ roberta-tagalog-large: 7718 entries


In [4]:
# Load Recon answer files (deduplicated)
RECON_MODELS = [
    'sealion-finetuned', 'sealion-vanilla',
    'llama-finetuned', 'llama-vanilla',
]

recon_answers = {}
for model in RECON_MODELS:
    path = f'../outputs/recon/{model}.jsonl'
    try:
        entries = [json.loads(l) for l in open(path)]
        # Build lookup by bare_sentence for fan-out to all benchmark entries
        recon_answers[model] = {e['bare_sentence']: e for e in entries}
        print(f'Recon {model}: {len(recon_answers[model])} unique sentences')
    except FileNotFoundError:
        print(f'Recon {model}: NOT FOUND')

Recon sealion-finetuned: 6113 unique sentences
Recon sealion-vanilla: 6113 unique sentences


Recon llama-finetuned: 6113 unique sentences
Recon llama-vanilla: 5136 unique sentences


## Metric Functions

In [5]:
import re
import unicodedata


def nfc(value):
    return unicodedata.normalize("NFC", value)


def unmark(value):
    return "".join(c for c in unicodedata.normalize("NFD", value)
                   if not unicodedata.category(c).startswith("M"))


def is_word_char(c):
    return unicodedata.category(c).startswith(('L', 'M'))


def find_whole_word(text, word):
    """Find all whole-word, case-insensitive matches. Returns (position, word)."""
    matches = []
    for m in re.finditer(re.escape(nfc(word)), nfc(text), re.IGNORECASE):
        start, end = m.start(), m.end()
        before_ok = start == 0 or not is_word_char(text[start - 1])
        after_ok = end == len(text) or not is_word_char(text[end])
        if before_ok and after_ok:
            matches.append((start, word))
    return matches


def get_variant_sequence(text, choices):
    """Get ordered list of whole-word variant occurrences (case-insensitive)."""
    occurrences = []
    for variant in choices:
        occurrences.extend(find_whole_word(text, variant))
    occurrences.sort(key=lambda x: x[0])
    return [v for _, v in occurrences]


def get_parsed_variant(answer):
    """Get parsed variant from answer, handling different field names."""
    return answer.get('parsed_variant') or answer.get('predicted_variant')


def compute_per_homograph_f1(predictions, gold_labels, homographs):
    """
    Compute F1 per homograph, averaged across its candidate classes.
    Returns dict of {homograph: f1_score}.
    """
    groups = defaultdict(lambda: {'pred': [], 'gold': []})
    for pred, gold, hg in zip(predictions, gold_labels, homographs):
        if pred is not None:
            groups[hg]['pred'].append(pred)
            groups[hg]['gold'].append(gold)
    
    f1s = {}
    for hg, data in groups.items():
        if len(data['pred']) == 0:
            f1s[hg] = 0.0
            continue
        labels = sorted(set(data['gold'] + data['pred']))
        if len(labels) <= 1:
            f1s[hg] = 1.0 if all(p == g for p, g in zip(data['pred'], data['gold'])) else 0.0
        else:
            f1s[hg] = f1_score(data['gold'], data['pred'], labels=labels, average='macro', zero_division=0)
    
    return f1s


def macro_f1(f1_per_homograph, homograph_subset=None):
    """Unweighted average of per-homograph F1."""
    if homograph_subset:
        vals = [f1_per_homograph[h] for h in homograph_subset if h in f1_per_homograph]
    else:
        vals = list(f1_per_homograph.values())
    return np.mean(vals) if vals else 0.0


def weighted_f1(f1_per_homograph, count_per_homograph, homograph_subset=None):
    """Weighted average of per-homograph F1 by instance count."""
    if homograph_subset:
        items = [(f1_per_homograph[h], count_per_homograph[h]) for h in homograph_subset if h in f1_per_homograph]
    else:
        items = [(f1, count_per_homograph[h]) for h, f1 in f1_per_homograph.items()]
    
    total_weight = sum(c for _, c in items)
    if total_weight == 0:
        return 0.0
    return sum(f1 * c for f1, c in items) / total_weight


def parse_rate(answers):
    """Percentage of responses where parsing changed the raw output."""
    total = len(answers)
    parsed = sum(1 for a in answers
                 if get_parsed_variant(a) is not None
                 and a.get('raw_output', '').strip() != get_parsed_variant(a))
    return parsed / total * 100 if total > 0 else 0.0

## MCQ Evaluation

In [6]:
# Reparse MCQ: check which choices appear as whole words in raw_output.
# If exactly one choice found → parsed. Zero or more than one → None (wrong).
for model, answers in mcq_answers.items():
    for a in answers:
        raw = a.get('raw_output', '')
        if not raw:
            continue
        answer = nfc(raw.strip())
        choices = a.get('choices', [])
        matches = [c for c in choices if find_whole_word(answer, c)]
        a['parsed_variant'] = matches[0] if len(matches) == 1 else None


def evaluate_mcq(model_name, answers):
    """Compute all MCQ metrics for a model."""
    predictions = [get_parsed_variant(a) for a in answers]
    gold_labels = [a['gold_variant'] for a in answers]
    homographs = [a['homograph'] for a in answers]
    
    f1s = compute_per_homograph_f1(predictions, gold_labels, homographs)
    
    counts = defaultdict(int)
    for h in homographs:
        counts[h] += 1
    
    total = len(answers)
    
    # Accuracy (Overall): correct out of all, unparseable = wrong
    overall_correct = sum(1 for p, g in zip(predictions, gold_labels) if p == g)
    
    # Accuracy (Unparsed): raw_output stripped vs gold_variant, no parsing
    unparsed_correct = sum(1 for a in answers if nfc(a.get('raw_output', '').strip()) == nfc(a['gold_variant']))
    
    # Valid Rate: % where raw_output.strip() or parsed_variant is one of the choices
    valid_count = 0
    for a in answers:
        choices = a.get('choices', [])
        raw = nfc(a.get('raw_output', '').strip())
        parsed = get_parsed_variant(a)
        if raw in [nfc(c) for c in choices] or parsed in choices:
            valid_count += 1
    
    # Timing
    times = [a['seconds'] for a in answers if 'seconds' in a]
    avg_time = np.mean(times) if times else 0.0
    total_time = sum(times)
    
    workable = ASYMMETRY_TIERS['Workable']
    moderate = ASYMMETRY_TIERS['Moderate']
    extreme = ASYMMETRY_TIERS['Extreme']
    work_mod = workable + moderate
    
    return {
        'Avg Time (s)': round(avg_time, 3),
        'Total Time (h)': round(total_time / 3600, 2),
        'Macro-F1 Overall': macro_f1(f1s),
        'Macro-F1 Workable': macro_f1(f1s, workable),
        'Macro-F1 Moderate': macro_f1(f1s, moderate),
        'Macro-F1 Extreme': macro_f1(f1s, extreme),
        'Macro-F1 Work+Mod': macro_f1(f1s, work_mod),
        'Macro-F1 Malumay': macro_f1(f1s, BIGKAS_GROUPS.get('malumay', [])),
        'Macro-F1 Mabilís': macro_f1(f1s, BIGKAS_GROUPS.get('mabilis', [])),
        'Macro-F1 Malumì': macro_f1(f1s, BIGKAS_GROUPS.get('malumi', [])),
        'Macro-F1 Maragsâ': macro_f1(f1s, BIGKAS_GROUPS.get('maragsa', [])),
        'Weighted F1 Overall': weighted_f1(f1s, counts),
        'Parse Rate': parse_rate(answers),
        'Valid Rate': valid_count / total * 100 if total else 0.0,
        'Accuracy (Overall)': overall_correct / total * 100 if total else 0.0,
        'Accuracy (Unparsed)': unparsed_correct / total * 100 if total else 0.0,
    }


mcq_results = {}
for model, answers in mcq_answers.items():
    mcq_results[model] = evaluate_mcq(model, answers)

mcq_df = pd.DataFrame(mcq_results).T
mcq_df = mcq_df.round(4)
print('=== MCQ Results ===')
mcq_df

=== MCQ Results ===


,Avg Time (s),Total Time (h),Macro-F1 Overall,Macro-F1 Workable,Macro-F1 Moderate,Macro-F1 Extreme,Macro-F1 Work+Mod,Macro-F1 Malumay,Macro-F1 Mabilís,Macro-F1 Malumì,Macro-F1 Maragsâ,Weighted F1 Overall,Parse Rate,Valid Rate,Accuracy (Overall),Accuracy (Unparsed)
sealion-finetuned,3.390,7.27,0.8271,0.8810,0.9026,0.7657,0.8886,0.8035,0.8096,0.9812,0.8197,0.8498,93.1718,93.2107,89.7512,0.0130
sealion-vanilla,1.553,3.33,0.2411,0.2605,0.3223,0.2001,0.2821,0.2543,0.2683,0.1055,0.1376,0.2283,0.0000,99.8963,43.9103,43.9103
llama-finetuned,3.760,8.06,0.8292,0.8799,0.8525,0.7880,0.8703,0.8067,0.7792,0.9657,0.8756,0.8269,69.2407,69.5906,65.7942,0.3110
llama-vanilla,1.792,3.84,0.2300,0.2843,0.2649,0.1825,0.2775,0.2610,0.2598,0.0220,0.1328,0.2236,0.1425,99.1578,38.0539,38.0539
roberta-base,0.000,0.00,0.5147,0.4409,0.4615,0.5814,0.4481,0.4761,0.4781,0.7241,0.4022,0.5043,0.0000,100.0000,73.7626,73.7626
roberta-large,0.000,0.00,0.4522,0.3424,0.4026,0.5409,0.3635,0.4249,0.4040,0.5856,0.3134,0.4450,0.0000,100.0000,69.7720,69.7720
roberta-tagalog-base,0.000,0.00,0.6012,0.6289,0.5169,0.6128,0.5897,0.5560,0.5806,0.7942,0.5719,0.5450,0.0000,100.0000,79.9819,79.9819
roberta-tagalog-large,0.000,0.00,0.6345,0.5814,0.5757,0.6896,0.5794,0.5966,0.5842,0.7503,0.5884,0.6140,0.0000,100.0000,92.6406,92.6406


## Recon Evaluation

In [7]:
# Group benchmark entries by bare_sentence to get all homographs per sentence
sentence_homographs = defaultdict(list)
for entry in benchmark:
    sentence_homographs[entry['bare_sentence']].append(entry)


def evaluate_recon(model_name, model_lookup):
    """
    Evaluate recon at the sentence level (deduplicated).
    Recovery: for every homograph in the sentence, the ordered sequence of
    whole-word variant occurrences (case-insensitive, including bare form)
    in the output must match the gold sentence.
    """
    total = len(model_lookup)
    recovered_count = 0
    exact_count = 0
    times = []

    for bare_sent, recon in model_lookup.items():
        raw_output = nfc(recon.get('raw_output', '').strip())
        entries = sentence_homographs.get(bare_sent, [])
        if not entries:
            continue

        if 'seconds' in recon:
            times.append(recon['seconds'])

        gold_sentence = nfc(entries[0]['sentence'].strip())

        # Exact match: full output == gold sentence
        if raw_output == gold_sentence:
            exact_count += 1

        # Recovery: check all homographs in this sentence
        all_match = True
        seen = set()
        for entry in entries:
            hg = entry['homograph']
            if hg in seen:
                continue
            seen.add(hg)
            gold_seq = get_variant_sequence(gold_sentence, entry['choices'])
            output_seq = get_variant_sequence(raw_output, entry['choices'])
            if gold_seq != output_seq or len(gold_seq) == 0:
                all_match = False
                break

        if all_match:
            recovered_count += 1

    avg_time = np.mean(times) if times else 0.0
    total_time = sum(times)

    return {
        'Avg Time (s)': round(avg_time, 3),
        'Total Time (h)': round(total_time / 3600, 2),
        'Recovery Rate': recovered_count / total * 100 if total else 0.0,
        'Exact Match': exact_count / total * 100 if total else 0.0,
    }


recon_results = {}
for model, lookup in recon_answers.items():
    recon_results[model] = evaluate_recon(model, lookup)

recon_df = pd.DataFrame(recon_results).T
recon_df = recon_df.round(4)
print('=== Recon Results (sentence-level) ===')
recon_df

=== Recon Results (sentence-level) ===


,Avg Time (s),Total Time (h),Recovery Rate,Exact Match
sealion-finetuned,7.166,12.17,96.9900,95.4687
sealion-vanilla,13.170,22.36,0.7361,0.1799
llama-finetuned,7.191,12.21,97.0391,95.5014
llama-vanilla,19.097,27.25,0.0195,0.0000


## Per-Homograph Breakdown

In [8]:
def per_homograph_table(model_name, answers):
    """Generate per-homograph F1 table for a single model."""
    predictions = [get_parsed_variant(a) for a in answers]
    gold_labels = [a['gold_variant'] for a in answers]
    homographs_list = [a['homograph'] for a in answers]
    
    f1s = compute_per_homograph_f1(predictions, gold_labels, homographs_list)
    
    rows = []
    for h in sorted(f1s.keys()):
        rows.append({
            'Homograph': h,
            'Asymmetry': ASYMMETRY_MAP.get(h, ''),
            'Bigkas': ', '.join(BIGKAS_TAGS_MAP.get(h, [])),
            'F1': round(f1s[h], 4),
        })
    
    return pd.DataFrame(rows).set_index('Homograph')


if mcq_answers:
    best_mcq = max(mcq_results.items(), key=lambda x: x[1]['Macro-F1 Overall'])
    print(f'Per-homograph F1 for best MCQ model: {best_mcq[0]}')
    display(per_homograph_table(best_mcq[0], mcq_answers[best_mcq[0]]))

Per-homograph F1 for best MCQ model: llama-finetuned


,Asymmetry,Bigkas,F1
Homograph,,,
abala,Moderate,"mabilis, malumay",0.6190
bagong,Extreme,malumay,1.0000
baka,Moderate,"maragsa, malumay",0.9524
bilang,Extreme,"mabilis, malumay",0.9817
buhay,Moderate,"mabilis, malumay",0.8886
bukas,Workable,"mabilis, malumay",0.8755
butas,Moderate,"mabilis, malumay",0.7302
dating,Moderate,"mabilis, malumay",0.9176
gabi,Extreme,"mabilis, malumay",0.4985


## Error Analysis by Bigkas Type

In [9]:
def error_analysis_bigkas(model_name, answers):
    """Show error patterns grouped by bigkas confusion type."""
    errors = [a for a in answers if get_parsed_variant(a) and get_parsed_variant(a) != a['gold_variant']]
    
    error_groups = defaultdict(list)
    for e in errors:
        error_groups[e['homograph']].append(e)
    
    sorted_errors = sorted(error_groups.items(), key=lambda x: len(x[1]), reverse=True)
    
    rows = []
    for hg, errs in sorted_errors[:15]:
        confusions = defaultdict(int)
        for e in errs:
            confusions[(e['gold_variant'], get_parsed_variant(e))] += 1
        
        top_confusion = max(confusions.items(), key=lambda x: x[1])
        gold, pred = top_confusion[0]
        
        meta = metadata.get(hg, {})
        gold_bigkas = meta.get('variants', {}).get(gold, {}).get('bigkas', '?')
        pred_bigkas = meta.get('variants', {}).get(pred, {}).get('bigkas', '?')
        
        rows.append({
            'Homograph': hg,
            'Errors': len(errs),
            'Top Confusion': f'{gold} -> {pred}',
            'Gold Bigkas': gold_bigkas,
            'Pred Bigkas': pred_bigkas,
            'Confusion Type': f'{gold_bigkas} -> {pred_bigkas}',
        })
    
    return pd.DataFrame(rows).set_index('Homograph')


if mcq_answers:
    best_model = best_mcq[0]
    print(f'Top 15 error sources for {best_model} (MCQ):')
    display(error_analysis_bigkas(best_model, mcq_answers[best_model]))

Top 15 error sources for llama-finetuned (MCQ):


,Errors,Top Confusion,Gold Bigkas,Pred Bigkas,Confusion Type
Homograph,,,,,
pangulong,121,pangúlong -> pàngulong,malumay,?,malumay -> ?
kilala,23,kilalá -> kilalâ,mabilis,?,mabilis -> ?
buhay,18,buháy -> búhay,mabilis,malumay,mabilis -> malumay
lalaki,12,lalakí -> laláki,mabilis,malumay,mabilis -> malumay
kabilang,11,kabilâng -> kabílang,maragsa,malumay,maragsa -> malumay
abala,9,abalá -> abalà,mabilis,?,mabilis -> ?
bukas,8,búkas -> bukás,malumay,mabilis,malumay -> mabilis
taong,8,taóng -> táong,mabilis,malumay,mabilis -> malumay
butas,8,butás -> bútas,mabilis,malumay,mabilis -> malumay


## Summary Tables

In [10]:
print('=' * 80)
print('MCQ RESULTS')
print('=' * 80)
display(mcq_df)

print()
print('=' * 80)
print('RECON RESULTS')
print('=' * 80)
display(recon_df)

MCQ RESULTS


,Avg Time (s),Total Time (h),Macro-F1 Overall,Macro-F1 Workable,Macro-F1 Moderate,Macro-F1 Extreme,Macro-F1 Work+Mod,Macro-F1 Malumay,Macro-F1 Mabilís,Macro-F1 Malumì,Macro-F1 Maragsâ,Weighted F1 Overall,Parse Rate,Valid Rate,Accuracy (Overall),Accuracy (Unparsed)
sealion-finetuned,3.390,7.27,0.8271,0.8810,0.9026,0.7657,0.8886,0.8035,0.8096,0.9812,0.8197,0.8498,93.1718,93.2107,89.7512,0.0130
sealion-vanilla,1.553,3.33,0.2411,0.2605,0.3223,0.2001,0.2821,0.2543,0.2683,0.1055,0.1376,0.2283,0.0000,99.8963,43.9103,43.9103
llama-finetuned,3.760,8.06,0.8292,0.8799,0.8525,0.7880,0.8703,0.8067,0.7792,0.9657,0.8756,0.8269,69.2407,69.5906,65.7942,0.3110
llama-vanilla,1.792,3.84,0.2300,0.2843,0.2649,0.1825,0.2775,0.2610,0.2598,0.0220,0.1328,0.2236,0.1425,99.1578,38.0539,38.0539
roberta-base,0.000,0.00,0.5147,0.4409,0.4615,0.5814,0.4481,0.4761,0.4781,0.7241,0.4022,0.5043,0.0000,100.0000,73.7626,73.7626
roberta-large,0.000,0.00,0.4522,0.3424,0.4026,0.5409,0.3635,0.4249,0.4040,0.5856,0.3134,0.4450,0.0000,100.0000,69.7720,69.7720
roberta-tagalog-base,0.000,0.00,0.6012,0.6289,0.5169,0.6128,0.5897,0.5560,0.5806,0.7942,0.5719,0.5450,0.0000,100.0000,79.9819,79.9819
roberta-tagalog-large,0.000,0.00,0.6345,0.5814,0.5757,0.6896,0.5794,0.5966,0.5842,0.7503,0.5884,0.6140,0.0000,100.0000,92.6406,92.6406



RECON RESULTS


,Avg Time (s),Total Time (h),Recovery Rate,Exact Match
sealion-finetuned,7.166,12.17,96.9900,95.4687
sealion-vanilla,13.170,22.36,0.7361,0.1799
llama-finetuned,7.191,12.21,97.0391,95.5014
llama-vanilla,19.097,27.25,0.0195,0.0000


## Export Results

In [11]:
mcq_df.to_csv('mcq_results.csv')
recon_df.to_csv('recon_results.csv')
print('Results saved to analysis/mcq_results.csv and analysis/recon_results.csv')

Results saved to analysis/mcq_results.csv and analysis/recon_results.csv
